# PySAD quick start

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/selimfirat/pysad/blob/master/examples/quickstart.ipynb)

This notebook detects anomalies in a data stream with [PySAD](https://github.com/selimfirat/pysad). It takes about a minute to run, and you don't need to install anything locally.

You will:

1. stream a real dataset one instance at a time,
2. score each instance as it arrives, while the model keeps learning from the stream,
3. compare a few online detectors,
4. turn raw scores into alerts with conformal calibration.

In [ ]:
%pip install -q pysad

## Get the data

We use the Arrhythmia dataset from [ODDS](http://odds.cs.stonybrook.edu/): 452 patients with 274 features each, where about 15% of the records are anomalies. We shuffle it once and then treat it as a stream.

In [ ]:
import os
import urllib.request

import numpy as np
from sklearn.utils import shuffle

from pysad.utils import Data

DATA_URL = "https://github.com/selimfirat/pysad/raw/master/examples/data/arrhythmia.mat"

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/arrhythmia.mat"):
    urllib.request.urlretrieve(DATA_URL, "data/arrhythmia.mat")

X_all, y_all = Data("data").get_data("arrhythmia.mat")
X_all, y_all = shuffle(X_all, y_all, random_state=61)
print(f"{X_all.shape[0]} instances, {X_all.shape[1]} features, {int(y_all.sum())} anomalies")

## Detect anomalies in the stream

Every PySAD model learns online. `fit_score_partial` learns from an instance and scores it in one call, so the model keeps adapting as the stream goes. The labels are only used to measure how well the detector did.

In [ ]:
from pysad.evaluation import AUROCMetric
from pysad.models import LODA
from pysad.utils import ArrayStreamer

np.random.seed(61)
model = LODA()
auroc = AUROCMetric()
scores = []

for x, y in ArrayStreamer(shuffle=False).iter(X_all, y_all):
    score = model.fit_score_partial(x)  # Learn from the instance, then score it.
    auroc.update(y, score)
    scores.append(score)

print(f"LODA AUROC: {auroc.get():.3f}")

## Compare detectors

Swapping detectors is a one-line change. Some models need the feature ranges up front; here we take them from the data for simplicity.

In [ ]:
from pysad.models import HalfSpaceTrees, xStream

feature_mins, feature_maxes = X_all.min(axis=0), X_all.max(axis=0)
detectors = {
    "LODA": lambda: LODA(),
    "Half-Space Trees": lambda: HalfSpaceTrees(
        feature_mins=feature_mins, feature_maxes=feature_maxes
    ),
    "xStream": lambda: xStream(),
}

for name, make_model in detectors.items():
    np.random.seed(61)
    detector, metric = make_model(), AUROCMetric()
    for x, y in ArrayStreamer(shuffle=False).iter(X_all, y_all):
        metric.update(y, detector.fit_score_partial(x))
    print(f"{name:>16}: AUROC {metric.get():.3f}")

## Look at the scores

Each dot is one instance in stream order. Red dots are the true anomalies, which the detector never saw labels for.

In [ ]:
import matplotlib.pyplot as plt

scores = np.array(scores)
steps = np.arange(len(scores))
anomalous = y_all == 1

plt.figure(figsize=(10, 3.5))
plt.scatter(steps[~anomalous], scores[~anomalous], s=8, color="tab:blue", label="normal")
plt.scatter(steps[anomalous], scores[anomalous], s=14, color="tab:red", label="anomaly")
plt.xlabel("position in the stream")
plt.ylabel("LODA anomaly score")
plt.legend()
plt.tight_layout()
plt.show()

## From scores to alerts

Raw scores have no fixed scale, so picking a threshold is hard. A conformal calibrator turns each score into a p-value: the share of recent scores that were at least as high. Alerting when p < 0.05 means "this is more unusual than 95% of the recent stream".

In [ ]:
from pysad.transform.probability_calibration import ConformalProbabilityCalibrator

np.random.seed(61)
model = LODA()
calibrator = ConformalProbabilityCalibrator(windowed=True, window_size=300)
alerts = []

for i, (x, y) in enumerate(ArrayStreamer(shuffle=False).iter(X_all, y_all)):
    p_value = calibrator.fit_transform_partial(model.fit_score_partial(x))
    if p_value < 0.05:
        alerts.append(i)

true_alerts = int(y_all[alerts].sum())
print(f"{len(alerts)} alerts, {true_alerts} of them true anomalies")

## Next steps

- Browse all detectors, transformers and evaluators in the [documentation](https://pysad.readthedocs.io/en/latest/).
- See the [examples](https://github.com/selimfirat/pysad/tree/master/examples) for ensembles, PyOD integration and more.
- Questions or ideas? Start a thread in [Discussions](https://github.com/selimfirat/pysad/discussions).